# Visualiser avec les Offsets Exacts
Utilise les positions caractères sauvegardées

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print("OK")

In [ ]:
import json

# Load corpus
print("[1] Load corpus...")
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"      {len(text):,} chars")

# Load boundary offsets
print("\n[2] Load boundary offsets...")
with open('results/camelbert_boundary_tokens_with_offsets.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

offsets = data['boundary_offsets']
metadata = data['metadata']
print(f"      {len(offsets):,} boundary tokens with offsets")

In [ ]:
# Build character highlight set
print("\n[3] Building character highlights...")

char_to_highlight = set()
for offset in offsets:
    for char_pos in range(offset['char_start'], offset['char_end']):
        char_to_highlight.add(char_pos)

print(f"      {len(char_to_highlight):,} characters to highlight")

In [ ]:
# Build HTML
print("\n[4] Building HTML...")

html_text = ""
for char_idx, char in enumerate(text):
    if char_idx in char_to_highlight:
        html_text += f'<span class="boundary">{char}</span>'
    else:
        if char == '&':
            html_text += '&amp;'
        elif char == '<':
            html_text += '&lt;'
        elif char == '>':
            html_text += '&gt;'
        else:
            html_text += char

html = f"""<!DOCTYPE html>
<html dir="rtl" lang="ar">
<head>
    <meta charset="UTF-8">
    <title>Boundary Tokens - Kitab Uqala</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Naskh+Arabic:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body {{
            direction: rtl;
            font-family: 'Noto Naskh Arabic', Arial, sans-serif;
            margin: 20px;
            background: #f5f5f5;
            line-height: 2;
        }}
        .container {{
            max-width: 1100px;
            margin: 0 auto;
            background: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1 {{ text-align: center; color: #333; }}
        .info {{ text-align: center; color: #666; padding: 10px; background: #f0f0f0; border-radius: 5px; margin: 20px 0; }}
        .stats {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin: 20px 0;
        }}
        .stat {{
            padding: 15px;
            background: #f9f9f9;
            border-left: 4px solid #ff8c00;
            border-radius: 3px;
        }}
        .stat-label {{ font-size: 12px; color: #666; }}
        .stat-value {{ font-size: 20px; font-weight: bold; color: #ff8c00; }}
        .text {{
            background: white;
            padding: 20px;
            border: 1px solid #ddd;
            border-radius: 5px;
            text-align: justify;
            margin: 20px 0;
            font-size: 16px;
        }}
        .boundary {{
            background: rgba(255, 165, 0, 0.3);
            border-left: 2px solid #ff8c00;
            padding: 1px 2px;
        }}
        .footer {{ text-align: center; color: #999; font-size: 12px; padding-top: 20px; border-top: 1px solid #ddd; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📚 Boundary Tokens - Kitab Uqala</h1>
        <div class="info">Caractères en <span style="background: rgba(255, 165, 0, 0.3); padding: 2px 4px;">orange</span> = boundary tokens</div>

        <div class="stats">
            <div class="stat">
                <div class="stat-label">Corpus</div>
                <div class="stat-value">{len(text):,}</div>
                <div class="stat-label">caractères</div>
            </div>
            <div class="stat">
                <div class="stat-label">Boundary Tokens</div>
                <div class="stat-value">{len(offsets):,}</div>
                <div class="stat-label">détectés</div>
            </div>
            <div class="stat">
                <div class="stat-label">Caractères</div>
                <div class="stat-value">{len(char_to_highlight):,}</div>
                <div class="stat-label">surlignés</div>
            </div>
            <div class="stat">
                <div class="stat-label">Pourcentage</div>
                <div class="stat-value">{metadata['boundary_percentage']:.2f}%</div>
                <div class="stat-label">du corpus</div>
            </div>
        </div>

        <div class="text">
            {html_text}
        </div>

        <div class="footer">
            <p>Source: camelbert_boundary_tokens_with_offsets.json</p>
            <p>Model: CAMeL-BERT</p>
            <p>Generated: 2026-04-21</p>
        </div>
    </div>
</body>
</html>"""

with open('results/visualization_boundary_tokens_final.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✓ HTML saved: results/visualization_boundary_tokens_final.html")

In [ ]:
from google.colab import files
files.download('results/visualization_boundary_tokens_final.html')
print("Downloaded!")